# MMA3001 — Pork Packaging Defect Detection Using RGB Images

**Method:** Supervised deep learning (transfer learning) using YOLOv8n object detection.  
**Dataset:** [Pork Rasher Error (Packaging), Roboflow Universe, version 4](https://universe.roboflow.com/hello-2aqe0/pork-rasher-error-packaging/dataset/4).  
**Classes:** `loose-meat`, `packaging-error`, `twisted-meat`, `unsealed`, `wrinkle`.

This notebook records the actual **416 px baseline**, **640 px resolution experiment**, and **held-out test evaluation**. Existing experiment outputs are retained from the original Colab run. The notebook was tidied for publication by removing redundant setup cells and clearing authentication output; **no training was rerun and no results were changed**.

### Scope
Detect and locate **visible defects** in RGB images of packaged pork. This does **not** determine microbiological safety or meat freshness. Dataset distribution, frame similarity, and missing examples of rare defect classes limit generalisation.

### How to reproduce
Run in Google Colab with a CUDA-capable GPU (the original experiment used Tesla T4). Run cells from top to bottom. When prompted, authenticate privately with Roboflow; **do not hard-code or publish your API key**. Training creates a temporary `/content/mma3001_runs` directory. Output is saved to Google Drive at the end.

**Recorded software environment:** Ultralytics 8.4.187; Python 3.13.16; PyTorch 2.11.0+cu130; Tesla T4 GPU. Roboflow SDK version was not captured.

## 1. GPU availability

Confirm that Colab has GPU access before training. **中文：** 训练前确认 GPU 已启用。

In [1]:
import torch

print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

GPU available: True
GPU name: Tesla T4


## 2. Install dependencies and download labelled RGB images

This cell installs `roboflow` and `ultralytics`, authenticates at runtime, and downloads dataset v4 as YOLOv8-format images and labels. The **authentication output has intentionally been removed** in this publication copy. **中文：** 登录过程的输出已经清除，确保不公开登录信息。

In [ ]:

!pip install -q roboflow ultralytics

import roboflow
from pathlib import Path

# Authenticate Roboflow
roboflow.login()

rf = roboflow.Roboflow()

# Access the dataset
project = rf.workspace("hello-2aqe0").project(
    "pork-rasher-error-packaging"
)

# Download Version 4 in YOLOv8 format
dataset = project.version(4).download("yolov8")

root = Path(dataset.location)

print("Dataset location:", root)
print("Dataset exists:", root.exists())
print("data.yaml exists:", (root / "data.yaml").exists())


## 3. Inspect data and configure YOLO

Train/validation/test: 3,349 / 120 / 80 images as recorded by the original run. Labels are YOLO bounding-box annotations; the model predicts defect class and location. **中文：** 这一步准备训练配置并统计类别。

In [5]:

import yaml

with open(root / "data.yaml") as f:
    data = yaml.safe_load(f)

print("Classes:", data["names"])

for split in ["train", "valid", "test"]:
    folder = root / split / "images"
    images = list(folder.glob("*")) if folder.exists() else []
    print(f"{split}: {len(images)} images")

# Prepare YOLO training configuration
data["path"] = str(root)
data["train"] = "train/images"
data["val"] = "valid/images"
data["test"] = "test/images"

config_path = "/content/mma3001_data.yaml"

with open(config_path, "w") as f:
    yaml.safe_dump(data, f)

print("Dataset ready for training!")



Classes: ['loose-meat', 'packaging-error', 'twisted-meat', 'unsealed', 'wrinkle']
train: 3349 images
valid: 120 images
test: 80 images
Dataset ready for training!


## 4. Class distribution (defect instances)

Counts describe **annotation instances**, not necessarily unique pictures. A single image can contain multiple annotations; blank images may contain none. Rare categories (`loose-meat`, `twisted-meat`, `wrinkle`) make class-level validation uncertain. **Important:** The supplied validation and test splits contain **zero labelled `wrinkle` instances**, so that class cannot be quantitatively evaluated on these splits.

In [8]:

from pathlib import Path
from collections import Counter
import yaml

root = Path("/content/Pork-Rasher-Error(Packaging)-4")

with open(root / "data.yaml") as f:
    data = yaml.safe_load(f)

names = data["names"]
names = [names[i] for i in sorted(names)] if isinstance(names, dict) else names

for split in ["train", "valid", "test"]:
    counts = Counter()

    for file in (root / split / "labels").glob("*.txt"):
        for line in file.read_text().splitlines():
            parts = line.split()
            if parts:
                counts[int(parts[0])] += 1

    print(f"\n{split.upper()}:")
    for i, name in enumerate(names):
        print(f"  {name}: {counts[i]} instances")



TRAIN:
  loose-meat: 66 instances
  packaging-error: 1712 instances
  twisted-meat: 72 instances
  unsealed: 1055 instances
  wrinkle: 18 instances

VALID:
  loose-meat: 3 instances
  packaging-error: 34 instances
  twisted-meat: 3 instances
  unsealed: 81 instances
  wrinkle: 0 instances

TEST:
  loose-meat: 3 instances
  packaging-error: 23 instances
  twisted-meat: 4 instances
  unsealed: 49 instances
  wrinkle: 0 instances


## 5. Baseline training: YOLOv8n at 416 pixels

Pretrained YOLOv8n weights are fine-tuned for five defect classes for **12 epochs**, batch 8, random seed 42. Losses include classification and bounding-box-related terms; weights are optimised by gradient-based backpropagation (AdamW in the training logs). The output below is the actual retained log from the original run.

In [6]:

from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=config_path,
    epochs=12,
    imgsz=416,
    batch=8,
    device=0,
    seed=42,
    plots=True,
    project="/content/mma3001_runs",
    name="yolov8n_baseline"
)


Ultralytics 8.4.187 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/mma3001_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=12, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_baseline, nbs=64, nms=None, opset=None, optimize=False,

## 6. Back up baseline outputs

Mount Google Drive and archive the baseline results before running the next experiment. The original run saved `yolov8n_baseline.zip`. **中文：** 备份模型权重、训练曲线和指标。

In [7]:

from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

save_dir = Path("/content/drive/MyDrive/MMA3001")
save_dir.mkdir(parents=True, exist_ok=True)

baseline_dir = Path("/content/mma3001_runs/yolov8n_baseline")

shutil.make_archive(
    str(save_dir / "yolov8n_baseline"),
    "zip",
    root_dir=str(baseline_dir)
)

print("Baseline saved successfully!")


Mounted at /content/drive
Baseline saved successfully!


## 7. Resolution experiment: YOLOv8n at 640 pixels

Use the **same architecture, dataset, 12 epochs, batch size, and random seed**; change input resolution from 416 to 640 pixels. This is a limited sensitivity/performance experiment rather than a proof that every aspect of performance improved.

In [9]:

from ultralytics import YOLO

improved_model = YOLO("yolov8n.pt")

improved_results = improved_model.train(
    data=config_path,
    epochs=12,
    imgsz=640,
    batch=8,
    device=0,
    seed=42,
    plots=True,
    project="/content/mma3001_runs",
    name="yolov8n_640"
)

print("Improved model training completed!")


Ultralytics 8.4.187 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/mma3001_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=12, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_640, nbs=64, nms=None, opset=None, optimize=False, opti

## 8. Final independent test evaluation

The 640 px model was selected using **validation** performance for precision/recall/mAP50. The following evaluation uses the held-out **test split** and saves reported results. Test results should not be used for further model selection or tuning.

In [10]:

from ultralytics import YOLO
from pathlib import Path
import shutil
import json

# Load improved model
model_path = "/content/mma3001_runs/yolov8n_640/weights/best.pt"
model = YOLO(model_path)

# Evaluate on independent test set
test_results = model.val(
    data=config_path,
    split="test",
    imgsz=640,
    device=0,
    plots=True,
    project="/content/mma3001_runs",
    name="final_test_640"
)

# Record evaluation metrics
metrics = {
    "precision": float(test_results.box.mp),
    "recall": float(test_results.box.mr),
    "mAP50": float(test_results.box.map50),
    "mAP50_95": float(test_results.box.map)
}

print("\nFINAL TEST RESULTS:")
for key, value in metrics.items():
    print(f"{key}: {value:.4f}")

# Save improved model and final test outputs
save_dir = Path("/content/drive/MyDrive/MMA3001")
save_dir.mkdir(parents=True, exist_ok=True)

for folder in ["yolov8n_640", "final_test_640"]:
    source = Path("/content/mma3001_runs") / folder
    if source.exists():
        shutil.make_archive(
            str(save_dir / folder),
            "zip",
            root_dir=str(source)
        )

with open(save_dir / "final_test_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("\nFinal test and model saved to Google Drive!")


Ultralytics 8.4.187 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,133,679 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1361.4±519.9 MB/s, size: 66.3 KB)
val: Scanning /content/Pork-Rasher-Error(Packaging)-4/test/labels... 80 images, 12 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 80/80 551.8it/s 0.1s
val: New cache created: /content/Pork-Rasher-Error(Packaging)-4/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.2it/s 4.0s
                   all         80         79      0.506      0.414      0.355      0.197
            loose-meat          2          3      0.237      0.237      0.142     0.0983
       packaging-error         16         23      0.237      0.478       0.28     0.0688
          twisted-meat          4          4          1          0      0.113     0.0287
              unsealed         49         4

## 9. Summary and limitations

### Recorded results (object detection, not classification accuracy)

| Metric | 416px baseline (validation) | 640px experiment (validation) | 640px selected model (test) |
|---|---:|---:|---:|
| Precision | 0.463 | 0.659 | 0.5063 |
| Recall | 0.386 | 0.486 | 0.4136 |
| mAP50 | 0.450 | 0.478 | 0.3553 |
| mAP50-95 | 0.288 | 0.261 | 0.1969 |

**Trade-off:** At 640 px, validation precision, recall, and mAP50 increased, but the more stringent mAP50-95 **decreased**; training duration increased from about 0.120 to 0.167 hours. The selected model achieved lower metrics on the held-out test split than on validation.

**Limitations:** Classes are highly imbalanced; `wrinkle` has no labelled validation or test instances. Only 3-4 examples of some rare classes are present in evaluation splits. Many images may be visually similar video frames; cross-split similarity has not been audited. These limitations restrict claims about real-world performance, particularly food safety and generalisation to new production environments.

**Model authorship and AI disclosure:** Ultralytics supplies the YOLOv8 implementation and pretrained weights. This project involved selecting the dataset, configuring and executing fine-tuning, comparing resolutions, evaluating results, and interpreting limitations. Generative AI was used for coding guidance, debugging, documentation, and analysis support; all material assistance should be acknowledged in the assessed report. The student must verify and explain the methods, results, and code.

**Remaining repository work:** Provide installation instructions, tests, source documentation, references and links to the model/result archives, and a report (5-10 pages).